In [56]:
import cv2
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

%matplotlib inline

In [49]:
def make_detector(name, nfeatures=5000):
    if name == 'SIFT':
        return cv2.SIFT_create(nfeatures=nfeatures)
    if name == 'ORB':
        return cv2.ORB_create(nfeatures=nfeatures)
    if name == 'AKAZE':
        return cv2.AKAZE_create()
    return None

In [55]:
def make_matcher(detector_name, matcher_name):
    if matcher_name == 'BF':
        if detector_name == 'SIFT' or detector_name == 'AKAZE':
            return cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        else:
            return cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    else:
        search_params = { 'checks': 50 }

        if detector_name == 'SIFT' or detector_name == 'AKAZE':
            index_params = { 'algorithm': 1, 'trees': 5 }
            return cv2.FlannBasedMatcher(index_params, search_params)
        else:
            index_params = {
                'algorithm': 6,
                'table_number': 6,
                'key_size': 12,
                'multi_probe_level': 1
            }
            return cv2.FlannBasedMatcher(index_params, search_params)

In [57]:
def load_images(path):
    exts = ['*.jpg', '*.jpeg', '*.png']
    names = []
    for e in exts:
        names += glob.glob(os.path.join(path, e))
    names = sorted(names) # "determinismo"
    imgs = [cv2.imread(n, cv2.IMREAD_COLOR) for n in names]
    return imgs, names

In [87]:
def extract_features(imgs, detector):
    kps, des, grays = [], [], []
    for im in imgs:
        g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        grays.append(g)
        kp, de = det.detectAndCompute(g, None)
        kps.append(kp)
        des.append(de)
    return kps, des, grays

In [83]:
def match_all(des_list, matcher, lowe_ratio=0.7):
    n = len(des_list)
    matches = {}
    for i in range(n):
        for j in range(i + 1, n):
            if des_list[i] is None or des_list[j] is None:
                continue
            raw = matcher.knnMatch(des_list[i], des_list[j], k=2)
            good = [m for m, n in raw if m.distance < lowe_ratio * n.distance]
            matches[(i, j)]=good
    return matches

In [80]:
imgs, names = load_images('../data/pato')
h, w = imgs[0].shape[:2]
K = np.array([[1.2 * max(w, h), 0, w / 2], [0, 1.2 * max(w, h), h / 2], [0, 0, 1]], float)

In [104]:
detector_name = 'SIFT'
matcher_name = 'FLANN'
min_matches = 10

In [88]:
det = make_detector(detector_name)
matcher = make_matcher(detector_name, matcher_name)
kps, des, grays = extract_features(imgs, det)
matches = match_all(des, matcher)